<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/ml_track_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🎯 **ML Track Exercises (Modules 17-22)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

# ML Track Exercises — Modules 17-22

Output-first practice for the Machine Learning & AI part of the course. Each exercise shows the expected output; you write the code; the answer is collapsed behind `👀 Show answer`.

### Setup


In [ ]:
!pip -q install torch torchvision scikit-learn pandas numpy matplotlib seaborn yfinance prophet pmdarima statsmodels

## Module 17 — Python for ML & AI

### 17.1 — Safe divide

Write `safe_divide(a, b)` that returns `None` if `b == 0`, otherwise `a / b`. Test on `(10, 2)`, `(10, 0)`, `(7, 3)`.

**Expected output / behaviour:**

```
5.0
None
2.3333333333333335
```


<details><summary>👀 <b>Show answer</b></summary>

```python
def safe_divide(a, b):
    return None if b == 0 else a / b

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(7, 3))
```

</details>


---

### 17.2 — Accuracy with a comprehension

Given `predictions = [1, 0, 1, 1, 0]` and `labels = [1, 1, 1, 0, 0]`, compute accuracy in one line using `sum()` + a comprehension. No NumPy.

**Expected output / behaviour:**

```
0.6
```


<details><summary>👀 <b>Show answer</b></summary>

```python
predictions = [1, 0, 1, 1, 0]
labels      = [1, 1, 1, 0, 0]
acc = sum(p == l for p, l in zip(predictions, labels)) / len(labels)
print(acc)
```

</details>


---

### 17.3 — Decorator that times any function

Write a `@timed` decorator that prints how long a function took. Apply to a function that sleeps 0.5s.

**Expected output / behaviour:**

```
slow_fn took 0.50s
Done
```


<details><summary>👀 <b>Show answer</b></summary>

```python
import time
def timed(fn):
    def wrap(*a, **kw):
        t0 = time.time()
        out = fn(*a, **kw)
        print(f"{fn.__name__} took {time.time()-t0:.2f}s")
        return out
    return wrap

@timed
def slow_fn():
    time.sleep(0.5)
    return "Done"

print(slow_fn())
```

</details>


---

### 17.4 — PyTorch tensor & gradient

Create `x = torch.tensor(3.0, requires_grad=True)`. Compute `y = x**2 + 2*x + 1`. Call `y.backward()`. Print `x.grad`.

**Expected output / behaviour:**

```
tensor(8.)
```


<details><summary>👀 <b>Show answer</b></summary>

```python
import torch
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1
y.backward()
print(x.grad)        # 2x + 2 = 8 at x=3
```

</details>


---

## Module 18 — Six Core Models

### 18.1 — Linear regression on California housing

Use `fetch_california_housing` + scikit-learn LinearRegression with 80/20 split (random_state=42). Print test R².

**Expected output / behaviour:**

```
0.5758  (approximately)
```


<details><summary>👀 <b>Show answer</b></summary>

```python
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X, y = fetch_california_housing(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
m = LinearRegression().fit(Xtr, ytr)
print(round(m.score(Xte, yte), 4))
```

</details>


---

### 18.2 — Logistic regression on breast cancer

Use `load_breast_cancer` + LogisticRegression(max_iter=5000) with 80/20 split (random_state=42). Print test accuracy.

**Expected output / behaviour:**

```
~0.97
```


<details><summary>👀 <b>Show answer</b></summary>

```python
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
X = StandardScaler().fit_transform(X)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
print(round(LogisticRegression(max_iter=5000).fit(Xtr, ytr).score(Xte, yte), 4))
```

</details>


---

### 18.3 — K-Means on 2D blobs

Generate 4 blobs with `make_blobs(n_samples=300, centers=4, random_state=0)`. Fit KMeans(n_clusters=4). Print inertia.

**Expected output / behaviour:**

```
(low number, around 200)
```


<details><summary>👀 <b>Show answer</b></summary>

```python
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
X, _ = make_blobs(n_samples=300, centers=4, random_state=0)
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)
print(round(km.inertia_, 2))
```

</details>


---

### 18.4 — Tiny MLP on Fashion-MNIST

Define a 2-layer MLP (784 → 128 → 10) with ReLU. Print the parameter count.

**Expected output / behaviour:**

```
101,770
```


<details><summary>👀 <b>Show answer</b></summary>

```python
import torch.nn as nn
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, 10),
)
print(f"{sum(p.numel() for p in mlp.parameters()):,}")
```

</details>


---

## Module 19 — Self-Attention

### 19.1 — Single-head self-attention forward shape

Implement a SelfAttention class with `d_model=4`. Run it on `(batch=1, seq=3, d=4)` random input. Print the output shape.

**Expected output / behaviour:**

```
torch.Size([1, 3, 4])
```


<details><summary>👀 <b>Show answer</b></summary>

```python
import torch, torch.nn as nn, torch.nn.functional as F
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_model, d_model, bias=False)
        self.v = nn.Linear(d_model, d_model, bias=False)
        self.d = d_model
    def forward(self, x):
        Q, K, V = self.q(x), self.k(x), self.v(x)
        s = (Q @ K.transpose(-2,-1)) / self.d**0.5
        return F.softmax(s, dim=-1) @ V

x = torch.randn(1, 3, 4)
print(SelfAttention(4)(x).shape)
```

</details>


---

### 19.2 — Verify softmax row-sums

Take the attention weight matrix from a SelfAttention forward pass. Verify each row sums to 1.

**Expected output / behaviour:**

```
tensor([1.0000, 1.0000, 1.0000])
```


<details><summary>👀 <b>Show answer</b></summary>

```python
attn = SelfAttention(4)
x = torch.randn(1, 3, 4)
Q, K = attn.q(x), attn.k(x)
s = (Q @ K.transpose(-2,-1)) / attn.d**0.5
w = F.softmax(s, dim=-1)[0]
print(w.sum(dim=-1).round(decimals=4))
```

</details>


---

## Module 20 — Multi-Head + Causal Attention

### 20.1 — MultiHead with d_model=8, n_heads=2

Build a MultiHeadAttention(d_model=8, n_heads=2). Pass `(batch=1, seq=4, d=8)` random input. Print output shape.

**Expected output / behaviour:**

```
torch.Size([1, 4, 8])
```


<details><summary>👀 <b>Show answer</b></summary>

```python
import torch, torch.nn as nn, torch.nn.functional as F
class MHA(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dh = h, d//h
        self.qkv = nn.Linear(d, 3*d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.h, self.dh).transpose(1, 2)
        k = k.view(B, T, self.h, self.dh).transpose(1, 2)
        v = v.view(B, T, self.h, self.dh).transpose(1, 2)
        s = q @ k.transpose(-2, -1) / self.dh**0.5
        o = F.softmax(s, dim=-1) @ v
        o = o.transpose(1, 2).contiguous().view(B, T, D)
        return self.out(o)

print(MHA(8, 2)(torch.randn(1, 4, 8)).shape)
```

</details>


---

### 20.2 — Causal mask peeking test

Define a causal-MHA. Mutate the LAST token of a `(1, 5, 8)` input. Verify positions 0-3 of the output don't change.

**Expected output / behaviour:**

```
max diff per position: [0.0, 0.0, 0.0, 0.0, X.X]
```


<details><summary>👀 <b>Show answer</b></summary>

```python
import torch
torch.manual_seed(0)
class CausalMHA(nn.Module):
    def __init__(self, d, h, T_max=128):
        super().__init__()
        self.h, self.dh = h, d//h
        self.qkv = nn.Linear(d, 3*d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
        self.register_buffer("mask",
            torch.tril(torch.ones(T_max, T_max)).view(1,1,T_max,T_max))
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B,T,self.h,self.dh).transpose(1,2)
        k = k.view(B,T,self.h,self.dh).transpose(1,2)
        v = v.view(B,T,self.h,self.dh).transpose(1,2)
        s = q @ k.transpose(-2,-1) / self.dh**0.5
        s = s.masked_fill(self.mask[:,:,:T,:T]==0, float("-inf"))
        o = F.softmax(s, dim=-1) @ v
        return self.out(o.transpose(1,2).contiguous().view(B, T, D))

m = CausalMHA(8, 2).eval()
x = torch.randn(1, 5, 8)
y1 = m(x.clone())
x_mod = x.clone(); x_mod[:, -1] = torch.randn(8)
y2 = m(x_mod)
print((y1 - y2).abs().sum(dim=-1).squeeze().tolist())
```

</details>


---

## Module 21 — Diffusion Models

### 21.1 — Forward-process variance check

With T=200 linear betas (1e-4..0.02), sample `x_0 = torch.zeros(1000, 2)`, jump to `t=199`, and verify `x_199` has approximately unit variance.

**Expected output / behaviour:**

```
variance ≈ 0.98 (close to 1)
```


<details><summary>👀 <b>Show answer</b></summary>

```python
import torch
T = 200
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
ab = torch.cumprod(alphas, dim=0)
x0 = torch.zeros(1000, 2)
eps = torch.randn_like(x0)
x_T = torch.sqrt(ab[-1]) * x0 + torch.sqrt(1 - ab[-1]) * eps
print(round(x_T.var().item(), 2))
```

</details>


---

### 21.2 — Predict-noise loss

For one batch `x0 = torch.randn(64, 2)` and a model that returns the input unchanged (`lambda x, t: x`), compute the MSE between the predicted noise (model output) and the true noise. Should be larger than zero.

**Expected output / behaviour:**

```
a positive number, e.g. 0.45
```


<details><summary>👀 <b>Show answer</b></summary>

```python
import torch, torch.nn.functional as F
T = 200
betas = torch.linspace(1e-4, 0.02, T); alphas = 1 - betas
ab = torch.cumprod(alphas, dim=0); sqrt_ab = torch.sqrt(ab); sqrt_1mab = torch.sqrt(1-ab)
x0 = torch.randn(64, 2)
t = torch.randint(0, T, (64,))
eps = torch.randn_like(x0)
xt = sqrt_ab[t][:, None] * x0 + sqrt_1mab[t][:, None] * eps
pred = xt   # the dumb "identity" model
print(round(F.mse_loss(pred, eps).item(), 4))
```

</details>


---

## Module 22 — Time-Series Forecasting

### 22.1 — ADF test on a random walk

Generate a random walk `np.cumsum(np.random.randn(500))`. Run the ADF test. Is it stationary? Then take `np.diff` and rerun. Now stationary?

**Expected output / behaviour:**

```
raw      ADF p≈0.5..1.0  → non-stationary
diff(1)  ADF p<0.05      → STATIONARY
```


<details><summary>👀 <b>Show answer</b></summary>

```python
import numpy as np
from statsmodels.tsa.stattools import adfuller
np.random.seed(0)
walk = np.cumsum(np.random.randn(500))
print("raw     :", round(adfuller(walk)[1], 4))
print("diff(1) :", round(adfuller(np.diff(walk))[1], 4))
```

</details>


---

### 22.2 — auto_arima on a synthetic series

Build a synthetic series with weekly seasonality (period=7). Fit `auto_arima(seasonal=True, m=7)`. Print the chosen `(p,d,q)` and `(P,D,Q,m)`.

**Expected output / behaviour:**

```
order=(p,d,q), seasonal_order=(P,D,Q,7)
```


<details><summary>👀 <b>Show answer</b></summary>

```python
import numpy as np, pandas as pd
from pmdarima import auto_arima
np.random.seed(0)
t = np.arange(365)
y = 50 + 0.05*t + 5*np.sin(2*np.pi*t/7) + np.random.normal(0, 1, 365)
ts = pd.Series(y)
m = auto_arima(ts, seasonal=True, m=7, suppress_warnings=True,
               stepwise=True, error_action="ignore")
print("order:", m.order, "  seasonal_order:", m.seasonal_order)
```

</details>


---

### 22.3 — Walk-forward MAE

Implement a `rolling_origin(series, model_fn, h=14, k=3)` that retrains 3 times on growing windows and returns an array of MAEs. Use auto_arima as the model.

**Expected output / behaviour:**

```
array([X, Y, Z]) — 3 MAE values
```


<details><summary>👀 <b>Show answer</b></summary>

```python
import numpy as np, pandas as pd
from pmdarima import auto_arima
np.random.seed(0)
ts = pd.Series(np.cumsum(np.random.randn(200)) + 50)

def rolling_origin(series, model_fn, h=14, k=3):
    out = []
    n = len(series) - k * h
    for i in range(k):
        end = n + i * h
        m = model_fn(series[:end])
        p = m.predict(n_periods=h)
        out.append(np.abs(p - series[end:end+h].values).mean())
    return np.array(out)

errs = rolling_origin(ts,
    lambda s: auto_arima(s, suppress_warnings=True, error_action="ignore", stepwise=True),
    h=14, k=3)
print(errs.round(2))
```

</details>


---

---
🎉 **You finished the ML track exercises.**

Numbers may differ from the expected output for runs that depend on:
- Live data (yfinance prices change)
- Random seeds not pinned across all libraries (PyTorch, sklearn, NumPy)
- Floating point past the 4th decimal

Match the *shape and order of magnitude*, not the exact value.